# Agentic RAG — Setup & Quick Start

本 Notebook 是 Agentic RAG 项目的入口点，带你完成：
1. 环境准备（GPU 验证 + 依赖安装）
2. 知识库文档加载
3. 向量索引 & 知识图谱构建
4. 三种 RAG 模式 Quick Demo
5. Agent 执行轨迹可视化
6. 知识图谱可视化


## 第一步：环境准备

In [ ]:
# GPU 验证
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

# 安装依赖
!pip install -q openai sentence-transformers faiss-cpu networkx numpy pandas matplotlib gradio pyyaml tqdm pymupdf

In [ ]:
import os
import sys

# 克隆项目（Colab中）
if not os.path.exists("agentic-rag"):
    !git clone https://github.com/XIECHENG6/agentic-rag.git
    # 或者直接上传项目文件

os.chdir("agentic-rag")
sys.path.insert(0, ".")

# API Key 配置
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
os.environ["OPENAI_API_BASE"] = "https://api.deepseek.com/v1"
print("API key configured \u2713")


## 第二步：加载知识库文档

In [ ]:
from data.documents import DOCUMENTS

print(f"知识库文档数: {len(DOCUMENTS)}")
for title, content in DOCUMENTS:
    print(f"  \U0001f4c4 {title}: {len(content)} 字符")


## 第三步：构建向量索引 + 知识图谱

In [ ]:
from src.pipeline import AgenticRAGPipeline

pipeline = AgenticRAGPipeline(verbose=True)

# 将文档注入pipeline
pipeline.ingest_texts(DOCUMENTS)

# 查看统计
stats = pipeline.stats()
print(f"\n\U0001f4ca Pipeline统计:")
for k, v in stats.items():
    print(f"  {k}: {v}")


## 第四步：Quick Demo — 三种模式对比

In [ ]:
demo_questions = [
    "QLoRA中使用的NF4量化和传统INT4量化有什么区别？",
    "ReAct框架和Chain-of-Thought的主要区别是什么？",
    "对比RAG中的Top-K检索和MMR检索的优缺点",
]

print("=" * 70)
for q in demo_questions:
    print(f"\n\u2753 问题: {q}")
    print("-" * 50)
    
    # Simple RAG
    simple = pipeline.simple_rag(q)
    print(f"\n\U0001f4ce Simple RAG:\n{simple['answer'][:200]}")
    
    # Hybrid RAG
    hybrid = pipeline.hybrid_rag(q)
    print(f"\n\U0001f517 Hybrid RAG:\n{hybrid['answer'][:200]}")
    
    # Agentic RAG (with trace)
    result = pipeline.ask(q, verbose=True)
    print(f"\n\U0001f916 Agentic RAG:\n{result['answer'][:200]}")
    print(f"   Type: {result['question_type']} | Strategy: {result['strategy']}")
    print(f"   Reformulations: {result['reformulations']}")
    print("=" * 70)


## 第五步：可视化 Agent 执行轨迹

In [ ]:
import json

# 选一个多跳问题展示完整trace
complex_q = "QLoRA的双重量化技术额外节省了多少显存？这个技术和NF4量化分别优化了模型权重的哪个方面？"
result = pipeline.ask(complex_q, verbose=False)

print(f"\u2753 问题: {complex_q}")
print(f"\n\U0001f916 答案:\n{result['answer']}")
print(f"\n\U0001f4ca 执行轨迹 ({len(result['trace'])} 步):")
for i, step in enumerate(result['trace'], 1):
    state = step.get('state', '?')
    print(f"\n  Step {i} [{state}]:")
    for k, v in step.items():
        if k != 'state' and isinstance(v, str) and len(v) > 200:
            print(f"    {k}: {v[:200]}...")
        elif k != 'state':
            print(f"    {k}: {v}")


## 第六步：知识图谱可视化

In [ ]:
import os
fig = pipeline.kg.to_matplotlib(figsize=(16, 12))
os.makedirs("results/figures", exist_ok=True)
fig.savefig("results/figures/knowledge_graph.png", dpi=150, bbox_inches="tight")
print("知识图谱已保存")


## 完成！下一步

- [02_Agent_vs_Simple_RAG.ipynb](02_Agent_vs_Simple_RAG.ipynb) — 完整基准测试（60题 × 4系统）
- [demo/app.py](../demo/app.py) — 交互式 Gradio 界面对比三种 RAG 系统
